# Publication Figures & Results Tables Evaluation

This notebook generates **6 publication-quality matplotlib figures** (300 DPI) and **4 results tables** from GEV (Generalized Extreme Value) experiments on word-order entropy and dependency distance distributions across Universal Dependencies treebanks.

**What this evaluation does:**
- Generates bar charts, scatter plots, mediation path diagrams, and bound-awareness panels
- Compiles summary statistics tables for GEV fit quality, regression coefficients, mediation analysis, and data quality
- Verifies 11 key statistics from the hypothesis against source data
- Builds 4 evaluation datasets (treebank visualization, figure quality, results tables, bound-awareness summary)

The demo uses a curated subset of ~68 treebanks and 60 bound-awareness examples.

In [1]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT on Colab, always install
_pip('loguru==0.7.3')

# Core packages — pre-installed on Colab, install locally only
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0', 'Pillow==11.3.0')


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import json
import math
import os
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
from loguru import logger

# ── Logging ──────────────────────────────────────────────────────────────────
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

1

## Data Loading

Load the curated demo dataset from GitHub (with local fallback).

In [3]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-42dac1-word-order-entropy-predicts-ordinal-tail/main/evaluation_iter3_publication_fig/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [4]:
data = load_data()

# Unpack data into the structures the original eval.py expects
exp1_meta = data["exp1_metadata"]
exp1_examples = data["exp1_examples"]
exp2_data = data["exp2_data"]
exp4_meta = data["exp4_metadata"]

logger.info(f"Loaded {len(exp1_examples)} treebank examples")
logger.info(f"Loaded {len(exp2_data['datasets'][0]['examples'])} bound-awareness examples")
logger.info(f"Metadata keys: {list(exp1_meta.keys())}")

10:02:25|INFO   |Loaded 68 treebank examples


10:02:25|INFO   |Loaded 60 bound-awareness examples


10:02:25|INFO   |Metadata keys: ['n_treebanks_analysed', 'fit_quality', 'dual_track', 'regression', 'mediation', 'evt_unique_pairs', 'spoken_written', 'grambank_crossval', 'discordant_languages']


## Configuration

Tunable parameters and constants used across figures and tables.

In [5]:
# ── Figure output settings ───────────────────────────────────────────────────
FIGURE_DPI = 300              # Publication quality (original: 300)
SAVE_FIGURES = False          # Don't save to disk in demo; display inline instead

# ── Nature-style color palette ───────────────────────────────────────────────
FAMILY_COLORS = {
    "Indo-European": "#E64B35",
    "Afro-Asiatic": "#4DBBD5",
    "Uralic": "#00A087",
    "Turkic": "#3C5488",
    "Sino-Tibetan": "#F39B7F",
    "Other": "#8491B4",
}

FAMILY_ORDER = ["Indo-European", "Afro-Asiatic", "Uralic", "Turkic", "Sino-Tibetan", "Other"]

DISCORDANT_TBS = {"ar_padt", "zh_gsd", "eu_bdt", "en_ewt", "tr_imst", "hi_hdtb"}
DISCORDANT_LABELS = {
    "ar_padt": "Arabic",
    "zh_gsd": "Chinese",
    "eu_bdt": "Basque",
    "en_ewt": "English",
    "tr_imst": "Turkish",
    "hi_hdtb": "Hindi",
}

# Map language families to color groups
FAMILY_MAP = {
    "Indo-European": "Indo-European",
    "Afro-Asiatic": "Afro-Asiatic",
    "Uralic": "Uralic",
    "Turkic": "Turkic",
    "Sino-Tibetan": "Sino-Tibetan",
}

# ── Bound-awareness settings ────────────────────────────────────────────────
BOUND_AWARENESS_LENGTHS = [10, 12, 14, 16, 18, 20]  # Sentence lengths to analyze
VIABILITY_THRESHOLD = 0.25                            # Null xi range threshold

## Helper Functions

Utility functions for parsing prediction fields and assigning family color groups.

In [6]:
def assign_family_group(family: str) -> str:
    """Assign a language family to one of the 6 color groups."""
    return FAMILY_MAP.get(family, "Other")


def parse_predict_field(field_str: str) -> dict:
    """Parse a JSON-encoded predict field string."""
    if isinstance(field_str, dict):
        return field_str
    try:
        return json.loads(field_str)
    except (json.JSONDecodeError, TypeError):
        return {}

## Figure 1: GEV Fit Quality

AIC comparison bar chart showing GEV model selection rates and goodness-of-fit pass rates.

In [7]:
logger.info("Generating Figure 1: GEV Fit Quality")
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel A: AIC best percentage
fq = exp1_meta["fit_quality"]
pct_gev = fq["pct_gev_aic_best"]
pct_other = 100 - pct_gev
bars = axes[0].bar(["GEV", "Other\n(Lognormal/Gamma)"], [pct_gev, pct_other],
                   color=["#E64B35", "#8491B4"], edgecolor="black", linewidth=0.5)
axes[0].set_ylabel("Percentage of bin-treebank combos (%)", fontsize=11)
axes[0].set_title("A. AIC Model Selection", fontsize=12, fontweight="bold")
axes[0].set_ylim(0, 105)
axes[0].bar_label(bars, fmt="%.1f%%", fontsize=10, padding=3)
axes[0].spines["top"].set_visible(False)
axes[0].spines["right"].set_visible(False)

# Panel B: GoF pass rates
pct_ad = fq["pct_ad_pass"]
pct_ks = 100 - pct_ad
bars2 = axes[1].bar(["AD/KS Pass", "AD/KS Fail"], [pct_ad, pct_ks],
                    color=["#00A087", "#F39B7F"], edgecolor="black", linewidth=0.5)
axes[1].set_ylabel("Percentage of combos (%)", fontsize=11)
axes[1].set_title("B. Goodness-of-Fit (AD/KS Test)", fontsize=12, fontweight="bold")
axes[1].set_ylim(0, 105)
axes[1].bar_label(bars2, fmt="%.1f%%", fontsize=10, padding=3)
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)

axes[1].annotate(
    f"N = {fq['n_combos']} combos",
    xy=(0.95, 0.95), xycoords="axes fraction",
    ha="right", va="top", fontsize=9, fontstyle="italic",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="wheat", alpha=0.5)
)

plt.tight_layout()
plt.show()
plt.close(fig)
logger.info("  Figure 1 complete: 4 bars, 2 panels")

10:02:25|INFO   |Generating Figure 1: GEV Fit Quality


10:02:25|INFO   |  Figure 1 complete: 4 bars, 2 panels


## Figure 2: Dual-Track Robustness

Scatter plot of raw vs normalized GEV shape parameter (xi), colored by language family.

In [8]:
logger.info("Generating Figure 2: Dual-Track Robustness")
xi_raw_list, xi_norm_list, families = [], [], []
for ex in exp1_examples:
    pred = parse_predict_field(ex.get("predict_our_method", "{}"))
    xr, xn = pred.get("xi_raw"), pred.get("xi_norm")
    if xr is not None and xn is not None:
        xi_raw_list.append(float(xr))
        xi_norm_list.append(float(xn))
        families.append(assign_family_group(ex.get("metadata_family", "Other")))

xi_raw = np.array(xi_raw_list)
xi_norm = np.array(xi_norm_list)

fig, ax = plt.subplots(figsize=(7, 7))
for fam in FAMILY_ORDER:
    mask = [f == fam for f in families]
    if any(mask):
        ax.scatter(xi_raw[mask], xi_norm[mask],
                   c=FAMILY_COLORS[fam], label=fam, alpha=0.7, s=30,
                   edgecolors="white", linewidth=0.3)

lims = [min(xi_raw.min(), xi_norm.min()) - 0.05, max(xi_raw.max(), xi_norm.max()) + 0.05]
ax.plot(lims, lims, "k--", alpha=0.5, linewidth=1, label="y = x")
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("$\\xi_{raw}$ (raw max-DD)", fontsize=12)
ax.set_ylabel("$\\xi_{norm}$ (normalized max-DD)", fontsize=12)
ax.set_title("Dual-Track GEV Shape Parameter Comparison", fontsize=13, fontweight="bold")
ax.legend(fontsize=8, loc="upper left", framealpha=0.8)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

ax.annotate(f"Spearman $\\rho$ = 0.9997",
    xy=(0.95, 0.05), xycoords="axes fraction", ha="right", va="bottom", fontsize=10,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", alpha=0.8))

plt.tight_layout(); plt.show(); plt.close(fig)
logger.info(f"  Figure 2 complete: {len(xi_raw)} points")

10:02:25|INFO   |Generating Figure 2: Dual-Track Robustness


10:02:25|INFO   |  Figure 2 complete: 68 points


## Figure 3: Bound-Awareness Simulation

Six panels showing null vs observed GEV xi by sentence length, with viability threshold assessment.

In [9]:
logger.info("Generating Figure 3: Bound-Awareness Simulation")
summary = exp2_data["metadata"]["summary_by_length"]
ba_examples = exp2_data["datasets"][0]["examples"]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes_flat = axes.flatten()
n_data_points = 0

for idx, length in enumerate(BOUND_AWARENESS_LENGTHS):
    ax = axes_flat[idx]
    slen = str(length)
    smry = summary[slen]

    null_xis, obs_xis, treebank_names = [], [], []
    for ex in ba_examples:
        if ex.get("metadata_sentence_length") == length:
            null_xis.append(float(ex["metadata_null_xi_lmom"]))
            obs_xis.append(float(ex["metadata_obs_xi_lmom"]))
            treebank_names.append(ex.get("metadata_treebank", ""))

    n_data_points += len(null_xis) + len(obs_xis)
    x_pos = np.arange(len(treebank_names))
    width = 0.35

    ax.bar(x_pos - width/2, null_xis, width, label="Null $\\xi$", color="#8491B4", alpha=0.8, edgecolor="black", linewidth=0.3)
    ax.bar(x_pos + width/2, obs_xis, width, label="Observed $\\xi$", color="#E64B35", alpha=0.8, edgecolor="black", linewidth=0.3)

    ax.set_xticks(x_pos)
    short_names = [t.split("_")[0][:3] for t in treebank_names]
    ax.set_xticklabels(short_names, rotation=45, fontsize=7, ha="right")
    ax.set_ylabel("$\\xi$ (GEV shape)", fontsize=9)

    rec = smry["recommendation"]
    track_label = "NORM" if "normalized" in rec else "RAW"
    null_range = smry["null_xi_range"]
    color = "#00A087" if null_range >= VIABILITY_THRESHOLD else "#E64B35"
    ax.set_title(f"n = {length}  |  null range = {null_range:.3f}  |  {track_label}",
                 fontsize=10, fontweight="bold", color=color)
    ax.axhline(y=0, color="gray", linestyle="-", linewidth=0.5, alpha=0.5)
    if idx == 0:
        ax.legend(fontsize=7, loc="lower left")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

fig.suptitle("Bound-Awareness: Null vs Observed GEV $\\xi$ by Sentence Length",
             fontsize=14, fontweight="bold", y=1.02)
fig.text(0.5, -0.02,
         f"Viability threshold: null $\\xi$ range > {VIABILITY_THRESHOLD}. Crossed at n = 14.",
         ha="center", fontsize=10, fontstyle="italic")

plt.tight_layout(); plt.show(); plt.close(fig)
logger.info(f"  Figure 3 complete: {n_data_points} points, 6 panels")

10:02:25|INFO   |Generating Figure 3: Bound-Awareness Simulation


10:02:26|INFO   |  Figure 3 complete: 120 points, 6 panels


## Figure 4: Regression Scatter

Word-order entropy vs GEV xi with language family coloring, regression line, and discordant language annotations.

In [10]:
logger.info("Generating Figure 4: Regression Scatter")
regression_meta = exp1_meta["regression"]
n_regression_tbs = regression_meta["n_treebanks"]

# Build dataframe from examples
rows = []
for ex in exp1_examples:
    pred = parse_predict_field(ex.get("predict_our_method", "{}"))
    xi_raw = pred.get("xi_raw")
    if xi_raw is None:
        continue
    rows.append({
        "treebank_id": ex.get("input", ex.get("metadata_treebank_id", "")),
        "xi_raw": float(xi_raw),
        "word_order_entropy": float(ex.get("metadata_word_order_entropy", 0)),
        "morph_richness": float(ex.get("metadata_morph_richness", 0)),
        "family": ex.get("metadata_family", "Other"),
        "family_group": assign_family_group(ex.get("metadata_family", "Other")),
        "n_bins": int(ex.get("metadata_n_bins", 0)),
    })

df = pd.DataFrame(rows)
df_reg = df[df["word_order_entropy"] > 0].copy()

fig, ax = plt.subplots(figsize=(10, 7))
n_data_points = 0
families_used = set()
for fam in FAMILY_ORDER:
    mask = df_reg["family_group"] == fam
    subset = df_reg[mask]
    if len(subset) > 0:
        families_used.add(fam)
        ax.scatter(subset["word_order_entropy"], subset["xi_raw"],
                   c=FAMILY_COLORS[fam], label=fam, alpha=0.7, s=40,
                   edgecolors="white", linewidth=0.3, zorder=3)
        n_data_points += len(subset)

# Regression line
coefs = regression_meta["coefficients"]
beta_woe = coefs["word_order_entropy_z"]["beta"]
p_fdr = coefs["word_order_entropy_z"]["p_fdr"]
partial_r = regression_meta["partial_correlations"]["word_order_entropy_z"]["r"]

x_vals = df_reg["word_order_entropy"].values
y_vals = df_reg["xi_raw"].values
if len(x_vals) > 2:
    z = np.polyfit(x_vals, y_vals, 1)
    x_line = np.linspace(x_vals.min(), x_vals.max(), 100)
    y_line = np.polyval(z, x_line)
    ax.plot(x_line, y_line, "k-", linewidth=2, alpha=0.7, zorder=4)

# Annotate discordant languages
n_discordant_annotated = 0
for _, row in df_reg.iterrows():
    tb_id = row["treebank_id"]
    if tb_id in DISCORDANT_TBS:
        label = DISCORDANT_LABELS[tb_id]
        ax.annotate(label, xy=(row["word_order_entropy"], row["xi_raw"]),
            xytext=(10, 10), textcoords="offset points", fontsize=8, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color="black", lw=0.8),
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.8, edgecolor="gray"),
            zorder=5)
        n_discordant_annotated += 1

ax.set_xlabel("Word-Order Entropy", fontsize=12)
ax.set_ylabel("$\\xi_{raw}$ (GEV shape parameter)", fontsize=12)
ax.set_title(f"Mixed-Effects Regression: Word-Order Entropy vs GEV $\\xi$ (n = {n_data_points})",
             fontsize=13, fontweight="bold")

stats_text = (f"$\\beta$ = {beta_woe:.3f}\n$p_{{FDR}}$ = {p_fdr:.4f}\n"
              f"partial $r$ = {partial_r:.3f}\npseudo-$R^2$ = {regression_meta['pseudo_r2']:.3f}")
ax.annotate(stats_text, xy=(0.02, 0.02), xycoords="axes fraction",
    fontsize=9, verticalalignment="bottom",
    bbox=dict(boxstyle="round,pad=0.4", facecolor="lightyellow", alpha=0.9, edgecolor="gray"))

ax.legend(fontsize=8, loc="upper left", framealpha=0.8)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout(); plt.show(); plt.close(fig)
logger.info(f"  Figure 4 complete: {n_data_points} pts, {n_discordant_annotated} discordant")

10:02:26|INFO   |Generating Figure 4: Regression Scatter


10:02:26|INFO   |  Figure 4 complete: 68 pts, 6 discordant


## Figure 5: Mediation Path Diagram

Preacher-Hayes mediation analysis showing indirect (morphological richness -> word-order entropy -> GEV xi) and direct paths.

In [11]:
logger.info("Generating Figure 5: Mediation Path Diagram")
med = exp1_meta["mediation"]

fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(0, 10); ax.set_ylim(0, 7); ax.axis("off")

# Boxes
box_style = dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor="black", linewidth=1.5)
ax.text(1.5, 5.5, "Morphological\nRichness\n(X)", ha="center", va="center", fontsize=12,
        fontweight="bold", bbox=box_style)
ax.text(5, 5.5, "Word-Order\nEntropy\n(M)", ha="center", va="center", fontsize=12,
        fontweight="bold", bbox=box_style)
ax.text(8.5, 5.5, "GEV $\\xi$\n(Y)", ha="center", va="center", fontsize=12,
        fontweight="bold", bbox=box_style)

# Indirect path arrows
ind_mean = med["indirect_effect_mean"]
ind_ci = med["indirect_effect_ci"]
ind_sig = med["indirect_significant"]

ax.annotate("", xy=(3.5, 5.5), xytext=(2.8, 5.5),
            arrowprops=dict(arrowstyle="->", color="#00A087", lw=2.5))
ax.text(3.15, 6.1, "a path", fontsize=9, ha="center", color="#00A087", fontweight="bold")
ax.annotate("", xy=(7.2, 5.5), xytext=(6.3, 5.5),
            arrowprops=dict(arrowstyle="->", color="#00A087", lw=2.5))
ax.text(6.75, 6.1, "b path", fontsize=9, ha="center", color="#00A087", fontweight="bold")

# Direct path
dir_mean = med["direct_effect_mean"]
dir_ci = med["direct_effect_ci"]
dir_sig = med["direct_significant"]

ax.annotate("", xy=(7.2, 4.2), xytext=(2.8, 4.2),
            arrowprops=dict(arrowstyle="->", color="#E64B35", lw=2, linestyle="dashed"))
ax.text(5, 3.7, f"Direct (c') = {dir_mean:.4f}\nCI [{dir_ci[0]:.4f}, {dir_ci[1]:.4f}]\n{'Significant' if dir_sig else 'Not significant'}",
        fontsize=9, ha="center", va="top",
        color="#E64B35" if dir_sig else "gray",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="#FFF5F5" if not dir_sig else "#FFE0E0", alpha=0.8))

# Indirect effect box
ax.text(5, 2.0,
        f"Indirect (a$\\times$b) = {ind_mean:.4f}\nCI [{ind_ci[0]:.4f}, {ind_ci[1]:.4f}]\n{'Significant' if ind_sig else 'Not significant'}",
        fontsize=10, ha="center", va="center", fontweight="bold", color="#00A087",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="#E8F8F5", alpha=0.9, edgecolor="#00A087", linewidth=1.5))

# Interpretation
total = med["total_effect_mean"]
interp = med["interpretation"]
ax.text(5, 0.8,
        f"Total effect = {total:.4f}  |  Interpretation: {interp.replace('_', ' ').title()}\n"
        f"n = {med['n_valid_bootstraps']} bootstraps",
        fontsize=9, ha="center", va="center", fontstyle="italic",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="wheat", alpha=0.5))

ax.set_title("Preacher-Hayes Mediation Analysis", fontsize=14, fontweight="bold", pad=20)
plt.tight_layout(); plt.show(); plt.close(fig)
logger.info("  Figure 5 complete")

10:02:26|INFO   |Generating Figure 5: Mediation Path Diagram


10:02:26|INFO   |  Figure 5 complete


## Figure 6: EVT Uniqueness & Spoken/Written Comparison

Pie chart of EVT-unique treebank pairs and bar chart comparing spoken vs written treebanks' GEV xi.

In [12]:
logger.info("Generating Figure 6: EVT Uniqueness & Spoken/Written")
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Panel A: EVT-unique pairs
evt = exp1_meta["evt_unique_pairs"]
pct_unique = evt["pct_evt_unique"]
pct_non_unique = 100 - pct_unique
sizes = [pct_unique, pct_non_unique]
labels = [f"EVT-unique\n{pct_unique:.1f}%", f"Non-unique\n{pct_non_unique:.1f}%"]
colors = ["#E64B35", "#8491B4"]
wedges, texts = axes[0].pie(sizes, labels=labels, colors=colors, startangle=90,
                             textprops={"fontsize": 11})
axes[0].set_title("A. EVT-Unique Treebank Pairs\n(among similar mean-DD pairs)",
                  fontsize=11, fontweight="bold")
axes[0].annotate(
    f"n_similar = {evt['n_similar_mean_dd']:,}\nn_unique = {evt['n_evt_unique']:,}",
    xy=(0.5, -0.1), xycoords="axes fraction", ha="center", fontsize=9, fontstyle="italic")

# Panel B: Spoken vs Written comparison
sw = exp1_meta["spoken_written"]
langs = [item["language"] for item in sw]
xi_spoken = [item["xi_spoken"] for item in sw]
xi_written = [item["xi_written"] for item in sw]

x_pos = np.arange(len(langs))
width = 0.35
axes[1].bar(x_pos - width/2, xi_spoken, width, label="Spoken", color="#F39B7F", edgecolor="black", linewidth=0.5)
axes[1].bar(x_pos + width/2, xi_written, width, label="Written", color="#3C5488", edgecolor="black", linewidth=0.5)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(langs, fontsize=9)
axes[1].set_ylabel("$\\xi_{raw}$", fontsize=11)
axes[1].set_title("B. Spoken vs Written Treebanks", fontsize=11, fontweight="bold")
axes[1].legend(fontsize=9)
axes[1].spines["top"].set_visible(False); axes[1].spines["right"].set_visible(False)

for i, item in enumerate(sw):
    diff = item["diff"]
    direction = "softer" if diff > 0 else "sharper"
    axes[1].annotate(f"$\\Delta$ = {diff:.3f}\n({direction})",
        xy=(x_pos[i], min(xi_spoken[i], xi_written[i]) - 0.02),
        fontsize=7, ha="center", va="top", fontstyle="italic")

plt.tight_layout(); plt.show(); plt.close(fig)
logger.info("  Figure 6 complete")

10:02:26|INFO   |Generating Figure 6: EVT Uniqueness & Spoken/Written


10:02:26|INFO   |  Figure 6 complete


## Results Tables

Compile 4 summary tables: GEV fit summary, regression coefficients, mediation results, and data quality.

In [13]:
logger.info("Compiling Tables")

# ── Table 1: GEV Fit Summary ────────────────────────────────────────────────
fq = exp1_meta["fit_quality"]
dt = exp1_meta["dual_track"]
evt = exp1_meta["evt_unique_pairs"]
reg = exp1_meta["regression"]

t1_rows = [
    {"Metric": "N bin-treebank combos", "Value": str(fq["n_combos"])},
    {"Metric": "GEV AIC-best (%)", "Value": f"{fq['pct_gev_aic_best']:.1f}"},
    {"Metric": "AD/KS pass rate (%)", "Value": f"{fq['pct_ad_pass']:.1f}"},
    {"Metric": "MLE-Lmom mean xi diff", "Value": f"{fq['mle_lmom_mean_diff']:.4f}"},
    {"Metric": "Dual-track Spearman rho", "Value": f"{dt['spearman_rho']:.4f}"},
    {"Metric": "Grambank cross-val Spearman r", "Value": f"{exp1_meta['grambank_crossval']['spearman_r']:.3f}"},
    {"Metric": "N treebanks analyzed", "Value": str(exp1_meta["n_treebanks_analysed"])},
    {"Metric": "N regression treebanks", "Value": str(reg["n_treebanks"])},
    {"Metric": "N language families", "Value": str(reg["n_families"])},
    {"Metric": "EVT-unique pairs (%)", "Value": f"{evt['pct_evt_unique']:.1f}"},
    {"Metric": "Baseline mean-DD R-sq", "Value": f"{reg['baseline_regression']['r_squared']:.3f}"},
    {"Metric": "GEV pseudo-R-sq", "Value": f"{reg['pseudo_r2']:.3f}"},
]
print("=== Table 1: GEV Fit Summary ===")
print(pd.DataFrame(t1_rows).to_string(index=False))

# ── Table 2: Regression Coefficients ────────────────────────────────────────
coefs = reg["coefficients"]
vif = reg["vif"]
partial = reg["partial_correlations"]
t2_rows = []
for predictor, vals in coefs.items():
    clean_name = predictor.replace("_z", "").replace("_", " ").title()
    t2_rows.append({
        "Predictor": clean_name, "Beta": f"{vals['beta']:.4f}", "SE": f"{vals['se']:.4f}",
        "p_FDR": f"{vals['p_fdr']:.6f}",
        "Sig (FDR)": "Yes" if vals["reject_fdr"] else "No",
        "VIF": f"{vif[predictor]:.2f}", "Partial r": f"{partial[predictor]['r']:.3f}",
    })
print("\n=== Table 2: Regression Coefficients ===")
print(pd.DataFrame(t2_rows).to_string(index=False))

# ── Table 3: Mediation Results ──────────────────────────────────────────────
med = exp1_meta["mediation"]
t3_rows = [
    {"Effect": "Indirect (a x b)", "Estimate": f"{med['indirect_effect_mean']:.4f}",
     "CI": f"[{med['indirect_effect_ci'][0]:.4f}, {med['indirect_effect_ci'][1]:.4f}]",
     "Significant": "Yes" if med["indirect_significant"] else "No"},
    {"Effect": "Direct (c')", "Estimate": f"{med['direct_effect_mean']:.4f}",
     "CI": f"[{med['direct_effect_ci'][0]:.4f}, {med['direct_effect_ci'][1]:.4f}]",
     "Significant": "Yes" if med["direct_significant"] else "No"},
    {"Effect": "Total (c)", "Estimate": f"{med['total_effect_mean']:.4f}", "CI": "--", "Significant": "--"},
    {"Effect": "Prop. mediated", "Estimate": f"{med['proportion_mediated']:.2f}", "CI": "--", "Significant": "--"},
    {"Effect": "Interpretation", "Estimate": med["interpretation"], "CI": "--", "Significant": "--"},
]
print("\n=== Table 3: Mediation Results ===")
print(pd.DataFrame(t3_rows).to_string(index=False))

# ── Table 4: Data Quality ───────────────────────────────────────────────────
gs = exp4_meta["global_summary"]
t4_rows = [
    {"Check": "Grambank-UD Spearman rho", "Value": str(gs.get("grambank_spearman_rho", "N/A")),
     "Status": gs.get("grambank_interpretation", "N/A")},
    {"Check": "Entropy threshold sensitivity", "Value": str(gs.get("entropy_spearman_10v20", "N/A")),
     "Status": "Robust" if gs.get("entropy_is_robust", False) else "Not robust"},
    {"Check": "Overall subset coverage", "Value": f"{gs.get('overall_coverage', 0):.4f}", "Status": "OK"},
    {"Check": "Autocorr proportion significant", "Value": f"{gs.get('autocorr_proportion_significant', 0):.4f}",
     "Status": "WARNING" if gs.get("autocorr_proportion_significant", 0) > 0.2 else "OK"},
    {"Check": "Annotation confound", "Value": str(gs.get("confound_rho", "N/A")),
     "Status": "CRITICAL" if gs.get("is_confound") == "True" else "OK"},
]
for i, flag in enumerate(gs.get("diagnostic_flags", [])):
    t4_rows.append({"Check": f"Diagnostic flag {i+1}", "Value": flag[:60], "Status": "FLAG"})
print("\n=== Table 4: Data Quality Summary ===")
print(pd.DataFrame(t4_rows).to_string(index=False))

logger.info("All 4 tables compiled")

10:02:26|INFO   |Compiling Tables


=== Table 1: GEV Fit Summary ===
                       Metric  Value
        N bin-treebank combos    918
             GEV AIC-best (%)   95.2
          AD/KS pass rate (%)   28.2
        MLE-Lmom mean xi diff 0.0286
      Dual-track Spearman rho 0.9997
Grambank cross-val Spearman r  0.340
         N treebanks analyzed    194
       N regression treebanks    172
          N language families     26
         EVT-unique pairs (%)   46.2
        Baseline mean-DD R-sq  0.048
              GEV pseudo-R-sq  0.163

=== Table 2: Regression Coefficients ===
           Predictor    Beta     SE    p_FDR Sig (FDR)  VIF Partial r
      Morph Richness -0.0271 0.0216 0.209586        No 1.36    -0.118
Head Direction Ratio -0.0311 0.0219 0.209586        No 1.10    -0.073
  Word Order Entropy  0.0840 0.0226 0.000587       Yes 1.38     0.246

=== Table 3: Mediation Results ===
          Effect       Estimate                CI Significant
Indirect (a x b)         0.0288  [0.0091, 0.0551]         Yes
    

## Key Statistics Verification

Verify all 11 critical statistics from the hypothesis match source data.

In [14]:
logger.info("Verifying key statistics...")

fq = exp1_meta["fit_quality"]
reg = exp1_meta["regression"]
med = exp1_meta["mediation"]
dt = exp1_meta["dual_track"]
evt = exp1_meta["evt_unique_pairs"]

checks = []
checks.append(("GEV AIC-best ~95.2%", abs(fq["pct_gev_aic_best"] - 95.2) < 0.5))
checks.append(("AD/KS pass ~28.2%", abs(fq["pct_ad_pass"] - 28.2) < 0.5))
woe_beta = reg["coefficients"]["word_order_entropy_z"]["beta"]
checks.append(("WOE beta ~0.084", abs(woe_beta - 0.084) < 0.002))
woe_pfdr = reg["coefficients"]["word_order_entropy_z"]["p_fdr"]
checks.append(("WOE p_FDR ~0.0006", abs(woe_pfdr - 0.000587) < 0.0002))
woe_pr = reg["partial_correlations"]["word_order_entropy_z"]["r"]
checks.append(("WOE partial_r ~0.246", abs(woe_pr - 0.246) < 0.005))
checks.append(("pseudo_R2 ~0.163", abs(reg["pseudo_r2"] - 0.163) < 0.005))
checks.append(("dual-track rho ~0.9997", abs(dt["spearman_rho"] - 0.9997) < 0.001))
checks.append(("EVT-unique ~46.2%", abs(evt["pct_evt_unique"] - 46.2) < 0.5))
ind_ci = med["indirect_effect_ci"]
checks.append(("med indirect CI lo ~0.009", abs(ind_ci[0] - 0.009) < 0.003))
checks.append(("med indirect CI hi ~0.055", abs(ind_ci[1] - 0.055) < 0.003))
dir_ci = med["direct_effect_ci"]
checks.append(("med direct CI crosses zero", dir_ci[0] < 0 < dir_ci[1]))

all_pass = True
for name, passed in checks:
    status = "PASS" if passed else "FAIL"
    logger.info(f"  {status}: {name}")
    if not passed:
        all_pass = False

n_passed = sum(p for _, p in checks)
logger.info(f"Key stats verified: {all_pass} ({n_passed}/{len(checks)} passed)")

10:02:26|INFO   |Verifying key statistics...


10:02:26|INFO   |  PASS: GEV AIC-best ~95.2%


10:02:26|INFO   |  PASS: AD/KS pass ~28.2%


10:02:26|INFO   |  PASS: WOE beta ~0.084


10:02:26|INFO   |  PASS: WOE p_FDR ~0.0006


...

10:02:26|INFO   |  PASS: EVT-unique ~46.2%


10:02:26|INFO   |  PASS: med indirect CI lo ~0.009


10:02:26|INFO   |  PASS: med indirect CI hi ~0.055


10:02:26|INFO   |  PASS: med direct CI crosses zero


10:02:26|INFO   |Key stats verified: True (11/11 passed)


## Results Summary

Aggregate metrics dashboard and xi distribution visualization across language families.

In [15]:
# ── Aggregate Metrics Summary ────────────────────────────────────────────────
print("=" * 60)
print("EVALUATION SUMMARY")
print("=" * 60)
summary_rows = [
    {"Metric": "Figures generated", "Value": "6/6"},
    {"Metric": "Tables compiled", "Value": "4/4"},
    {"Metric": "Key stats verified", "Value": f"{n_passed}/11"},
    {"Metric": "Treebanks in demo", "Value": str(len(exp1_examples))},
    {"Metric": "Bound-awareness examples", "Value": str(len(ba_examples))},
    {"Metric": "Regression treebanks (full)", "Value": str(reg["n_treebanks"])},
    {"Metric": "Language families", "Value": str(reg["n_families"])},
    {"Metric": "Discordant languages annotated", "Value": str(n_discordant_annotated)},
]
print(pd.DataFrame(summary_rows).to_string(index=False))

# ── Visualization: Xi distribution by family group ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Box plot of xi_raw by family group
plot_data = []
for ex in exp1_examples:
    pred = parse_predict_field(ex.get("predict_our_method", "{}"))
    xi = pred.get("xi_raw")
    if xi is not None:
        fg = assign_family_group(ex.get("metadata_family", "Other"))
        plot_data.append({"family_group": fg, "xi_raw": float(xi)})

df_plot = pd.DataFrame(plot_data)
groups_present = [f for f in FAMILY_ORDER if f in df_plot["family_group"].values]
box_data = [df_plot[df_plot["family_group"] == f]["xi_raw"].values for f in groups_present]
bp = axes[0].boxplot(box_data, tick_labels=groups_present, patch_artist=True, widths=0.6)
for patch, fam in zip(bp["boxes"], groups_present):
    patch.set_facecolor(FAMILY_COLORS[fam])
    patch.set_alpha(0.7)
axes[0].set_ylabel("$\\xi_{raw}$ (GEV shape)", fontsize=11)
axes[0].set_title("A. GEV Shape Parameter by Language Family", fontsize=12, fontweight="bold")
axes[0].tick_params(axis="x", rotation=30)
axes[0].spines["top"].set_visible(False); axes[0].spines["right"].set_visible(False)
axes[0].axhline(y=df_plot["xi_raw"].median(), color="gray", linestyle="--", alpha=0.5, label="Overall median")
axes[0].legend(fontsize=8)

# Panel B: Histogram of xi_raw values
axes[1].hist(df_plot["xi_raw"], bins=15, color="#E64B35", alpha=0.7, edgecolor="black", linewidth=0.5)
axes[1].axvline(x=df_plot["xi_raw"].median(), color="black", linestyle="--", linewidth=1.5,
                label=f"Median = {df_plot['xi_raw'].median():.3f}")
axes[1].set_xlabel("$\\xi_{raw}$ (GEV shape parameter)", fontsize=11)
axes[1].set_ylabel("Count", fontsize=11)
axes[1].set_title("B. Distribution of GEV Shape Parameters", fontsize=12, fontweight="bold")
axes[1].legend(fontsize=9)
axes[1].spines["top"].set_visible(False); axes[1].spines["right"].set_visible(False)

plt.tight_layout(); plt.show(); plt.close(fig)

print("\nAll figures generated, tables compiled, and statistics verified successfully.")

EVALUATION SUMMARY
                        Metric Value
             Figures generated   6/6
               Tables compiled   4/4
            Key stats verified 11/11
             Treebanks in demo    68
      Bound-awareness examples    60
   Regression treebanks (full)   172
             Language families    26
Discordant languages annotated     6



All figures generated, tables compiled, and statistics verified successfully.
